# 02. Trellis and E8 Lattice Quantization: QuIP# and QTIP

- **Category:** 14_quantisation / 04_vector_lattice_qat_and_lowbit
- **Lineage & Citations:** Marcellin & Fischer (1990, *Trellis Coded Quantization*); Viazovska (2017, *The Sphere Packing Problem in Dimension 8*); Tseng et al. (2024, *QuIP#: Even Better LLM Quantization with Hadamard Incoherence and Lattice Codebooks*); Tseng et al. (2024, *QTIP: Quantization with Trellis-Coded Information Packing*).
- **Purpose:** Replace slow codebook clustering with regular point lattices: leveraging Maryna Viazovska's optimal 8D $E_8$ Gosset lattice packing and Conway-Sloane closed-form $O(d)$ decoding to compress weights to 2 bits without codebook lookups.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [01. Vector Product and Additive Quantization VQ PQ RVQ AQLM](01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.ipynb) $\to$ `🔗 Downstream Syntheses:` [03. QAT STE LSQ LSQ+ PACT DoReFa](03_qat_ste_lsq_lsq_plus_pact_and_dorefa.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT apply $E_8$ lattice quantization directly to raw weight matrices with severe outliers without Hadamard rotation preprocessing (lattice boundaries saturate on heavy tails); do NOT use exhaustive nearest neighbor search when Conway-Sloane parity decoding yields exact projections in $O(d)$ time.

## Mathematical Foundations & Conway-Sloane E8 Lattice Decoding

1. **Gosset Lattice Union Definition**:
   $$E_8 = D_8 \cup \left( D_8 + \frac{1}{2}\mathbf{1} \right), \quad D_8 = \left\{ \mathbf{x} \in \mathbb{Z}^8 \;\middle|\; \sum_{i=1}^8 x_i \equiv 0 \pmod 2 \right\}$$
2. **Conway & Sloane $D_8$ Parity Correction**:
   $$\mathbf{f} = \lfloor \mathbf{x} \rceil, \quad s = \sum_{i=1}^8 f_i \pmod 2, \quad k = \arg\max_i |x_i - f_i|, \quad f_k \leftarrow f_k + \text{sign}(x_k - f_k) \text{ if } s = 1$$
3. **Dual-Coset Minimum Distance Selection**:
   $$\mathbf{p}_0 = \text{proj}_{D_8}(\mathbf{x}), \quad \mathbf{p}_1 = \text{proj}_{D_8}(\mathbf{x} - 0.5) + 0.5, \quad \text{proj}_{E_8}(\mathbf{x}) = \arg\min_{\mathbf{p} \in \{\mathbf{p}_0, \mathbf{p}_1\}} \|\mathbf{x} - \mathbf{p}\|_2$$
4. **Trellis-Coded Quantization (TCQ) Path Metric**:
   $$\mathcal{L} = \sum_{t=1}^L (w_t - \hat{w}_t)^2 \quad \text{minimized via dynamic programming Viterbi search}$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/04_vector_lattice_qat_and_lowbit/02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def project_d8(x: torch.Tensor):
    f = torch.round(x)
    s = torch.sum(f, dim=-1, keepdim=True).long()
    diff = x - f
    max_idx = torch.argmax(torch.abs(diff), dim=-1, keepdim=True)
    delta = torch.where(diff >= 0.0, 1.0, -1.0)
    mask = torch.zeros_like(x).scatter_(-1, max_idx, 1.0)
    return torch.where((s % 2 != 0) & (mask > 0.0), f + delta, f)

def project_e8(x: torch.Tensor):
    p0 = project_d8(x)
    p1 = project_d8(x - 0.5) + 0.5
    d0 = torch.sum((x - p0)**2, dim=-1, keepdim=True)
    d1 = torch.sum((x - p1)**2, dim=-1, keepdim=True)
    return torch.where(d0 <= d1, p0, p1)

def quantize_e8_lattice(vectors: torch.Tensor, scale: torch.Tensor):
    pts = project_e8(vectors / scale)
    return pts * scale, pts

def e8_forward(x: torch.Tensor, w_orig: torch.Tensor, scale: torch.Tensor, bias: torch.Tensor | None = None):
    d_in, d_out = w_orig.shape
    w_deq_vecs, _ = quantize_e8_lattice(w_orig.reshape(-1, 8), scale)
    w_deq = w_deq_vecs.reshape(d_in, d_out)
    y = x @ w_deq
    return ((y + bias) if bias is not None else y), w_deq

In [ ]:
# 1. Linear anchor parity
y = weights["test_input"] @ weights["weight"] + weights["bias"]
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-5

# 2. E8 Lattice quantization parity
w_deq, pts = quantize_e8_lattice(weights["w_vecs"], weights["e8_scale"])
assert float(torch.max(torch.abs(w_deq - weights["w_e8_deq"]))) < 1e-5
assert float(torch.max(torch.abs(pts - weights["lattice_points"]))) < 1e-5

# 3. E8 forward pass parity
out_e8, w_full = e8_forward(weights["x_calib"], weights["w_orig"], weights["e8_scale"], weights["b_orig"])
assert float(torch.max(torch.abs(w_full - weights["w_deq_full"]))) < 1e-5
assert float(torch.max(torch.abs(out_e8 - weights["out_e8"]))) < 1e-5
print("All E8 Gosset lattice parity assertions verified against golden checkpoint!")

In [ ]:
# Comparative SQNR Benchmark: E8 Lattice Reconstruction Quality
w_target = weights["w_vecs"]
sig_pwr = float(torch.mean(w_target ** 2))

def eval_sqnr(w_recon: torch.Tensor) -> float:
    mse = float(torch.mean((w_target - w_recon) ** 2))
    return 10.0 * torch.log10(torch.tensor(sig_pwr / (mse + 1e-12))).item()

sqnr_e8 = eval_sqnr(weights["w_e8_deq"])
print(f"E8 Gosset Lattice Reconstruction SQNR: {sqnr_e8:.2f} dB")
assert sqnr_e8 > 15.0, "E8 Gosset lattice must achieve >15 dB SQNR on calibrated scale!"

In [ ]:
# Latency Microbenchmark: Conway-Sloane Closed-Form E8 Lattice Decoding
n_vecs = 32768
vecs_mock = torch.randn(n_vecs, 8)
scale_mock = torch.tensor([[0.10]])

t0 = time.perf_counter()
for _ in range(50):
    _ = project_e8(vecs_mock)
t_e8 = (time.perf_counter() - t0) / 50 * 1e3

print(f"Conway-Sloane E8 Lattice Decoding (32768x8) latency: {t_e8:.3f} ms")
assert t_e8 < 25.0, "E8 lattice decoding must complete in <25 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Mathematical Optimality**: The $E_8$ Gosset lattice is provably the densest sphere packing in 8 dimensions, unlocking maximum space-filling efficiency.
2. **Zero Codebook Overhead**: Conway-Sloane decoding computes projections in closed-form $O(d)$ time without codebook lookups.
3. **Trellis Structure**: Pairing lattices with convolutional state machines (TCQ/QTIP) allows 2-bit quantization to approach the Shannon rate-distortion bound.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.md](../../../markdown/14_quantisation/04_vector_lattice_qat_and_lowbit/02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.md)
- **PyTorch 2.x Module:** [02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.py](../../../pytorch2/14_quantisation/04_vector_lattice_qat_and_lowbit/02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.py)
- **Functional JAX Module:** [02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.py](../../../jax/14_quantisation/04_vector_lattice_qat_and_lowbit/02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.py)
- **Native Rust Candle Track:** [02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.rs](../../../candle/14_quantisation/04_vector_lattice_qat_and_lowbit/02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.rs)
- **Hardware Kernel (CPU / WGSL):** [02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.rs](../../../kernels/14_quantisation/04_vector_lattice_qat_and_lowbit/02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.rs)
- **Golden SafeTensors:** [02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.safetensors](../../../weights/14_quantisation/04_vector_lattice_qat_and_lowbit/02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. Vector Product and Additive Quantization VQ PQ RVQ AQLM](01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 03. QAT STE LSQ LSQ+ PACT DoReFa](03_qat_ste_lsq_lsq_plus_pact_and_dorefa.ipynb) &rarr;